Dataset


In [1]:
import numpy as np
import pandas as pd
import datetime
import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Juegos exclusivos de carta
games = ['blackjack', 'poker_texas_holdem', 'poker_cards', 'baccarat', 'pontoon']

n_users = 1000
n_interactions = 25000
user_ids = [f'user_{i}' for i in range(1, n_users+1)]

start = datetime.datetime(2024,1,1)
end = datetime.datetime(2025,12,1)
seconds_range = int((end - start).total_seconds())

rows = []
for _ in range(n_interactions):
    u = random.choice(user_ids)
    g = random.choice(games)
    ts = start + datetime.timedelta(seconds=random.randint(0, seconds_range))
    base = {'blackjack':10,'poker_texas_holdem':50,'poker_cards':30,'baccarat':40,'pontoon':15}[g]
    bet = max(1, int(np.random.exponential(scale=base)))
    rows.append((u,g,ts,bet))

df = pd.DataFrame(rows, columns=['user_id','game','timestamp','bet_amount'])
df = df.sort_values('timestamp').reset_index(drop=True)
print('Interacciones:', len(df))
df.head()


Interacciones: 25000


,user_id,game,timestamp,bet_amount
0,user_443,poker_3cards,2024-01-01 00:57:37,34
1,user_557,pontoon,2024-01-01 01:18:11,5
2,user_564,pontoon,2024-01-01 01:35:32,5
3,user_337,poker_3cards,2024-01-01 02:45:03,121
4,user_571,pontoon,2024-01-01 03:09:49,19


Procesamiento

In [2]:
# Exploración rápida
print(df['game'].value_counts())
print(df['bet_amount'].describe())

# Feature: hora del día (podría ser útil más adelante)
df['hour'] = df['timestamp'].dt.hour
# Señal implícita simple basada en bet_amount
import numpy as np
df['weight'] = np.log1p(df['bet_amount'])

df.head()


game
baccarat              5098
pontoon               5002
poker_3cards          4982
blackjack             4976
poker_texas_holdem    4942
Name: count, dtype: int64
count    25000.000000
mean        28.691840
std         35.988869
min          1.000000
25%          6.000000
50%         16.000000
75%         37.000000
max        461.000000
Name: bet_amount, dtype: float64


,user_id,game,timestamp,bet_amount,hour,weight
0,user_443,poker_3cards,2024-01-01 00:57:37,34,0,3.555348
1,user_557,pontoon,2024-01-01 01:18:11,5,1,1.791759
2,user_564,pontoon,2024-01-01 01:35:32,5,1,1.791759
3,user_337,poker_3cards,2024-01-01 02:45:03,121,2,4.804021
4,user_571,pontoon,2024-01-01 03:09:49,19,3,2.995732


Procesamiento


In [3]:
import scipy.sparse as sp
user_map = {u:i for i,u in enumerate(df['user_id'].unique())}
item_map = {g:i for i,g in enumerate(df['game'].unique())}
inv_item_map = {i:g for g,i in item_map.items()}

rows = df['user_id'].map(user_map).to_numpy()
cols = df['game'].map(item_map).to_numpy()
vals = df['weight'].to_numpy()

mat = sp.coo_matrix((vals, (rows, cols)), shape=(len(user_map), len(item_map))).tocsr()
print('Matriz shape:', mat.shape, 'NNZ:', mat.nnz)

# Mostrar densidad por juego
pd.DataFrame({'game': list(item_map.keys()), 'counts': mat.sum(axis=0).A1}).sort_values('counts', ascending=False)


Matriz shape: (1000, 5) NNZ: 4965


,game,counts
4,poker_texas_holdem,16840.184396
2,baccarat,16244.184101
0,poker_3cards,14585.242754
1,pontoon,11618.679547
3,blackjack,9858.475115


Control de Gestion


In [4]:
def train_als(matrix, n_factors=12, n_iters=10, reg=0.1):
    users, items = matrix.shape
    U = 0.01 * np.random.randn(users, n_factors)
    V = 0.01 * np.random.randn(items, n_factors)
    for it in range(n_iters):
        # update U
        for u in range(users):
            idx = matrix[u].indices
            if len(idx) == 0: continue
            V_i = V[idx]
            A = V_i.T.dot(V_i) + reg * np.eye(n_factors)
            b = V_i.T.dot(matrix[u].data)
            U[u] = np.linalg.solve(A, b)
        # update V
        for i in range(items):
            col = matrix[:, i].tocoo()
            idx = col.row
            if len(idx) == 0: continue
            U_u = U[idx]
            A = U_u.T.dot(U_u) + reg * np.eye(n_factors)
            b = U_u.T.dot(col.data)
            V[i] = np.linalg.solve(A, b)
        if (it+1) % 5 == 0:
            print(f'iter {it+1}/{n_iters}')
    return U, V

U, V = train_als(mat, n_factors=12, n_iters=8, reg=0.1)
print('Entrenado: U', U.shape, 'V', V.shape)


iter 5/8
Entrenado: U (1000, 12) V (5, 12)


Recomendación

In [5]:
def recommend_for_user(u_index, U, V, known_items=set(), topk=5):
    scores = V.dot(U[u_index])
    for i in known_items:
        if 0 <= i < len(scores): scores[i] = -np.inf
    top_idx = np.argsort(scores)[::-1][:topk]
    return [inv_item_map[i] for i in top_idx]

# Ejemplo para un usuario aleatorio
sample_user = random.choice(list(user_map.keys()))
uidx = user_map[sample_user]
known = set(mat[uidx].indices)
print('Usuario:', sample_user)
print('Conocidos:', [inv_item_map[i] for i in known])
print('Recomendaciones:', recommend_for_user(uidx, U, V, known_items=known, topk=5))


Usuario: user_571
Conocidos: ['poker_3cards', 'pontoon', 'baccarat', 'blackjack', 'poker_texas_holdem']
Recomendaciones: ['poker_texas_holdem', 'blackjack', 'baccarat', 'pontoon', 'poker_3cards']


Evaluacion Precision@k


In [6]:
# Split temporal: última interacción por usuario -> test
df_sorted = df.sort_values('timestamp')
last_idx = df_sorted.groupby('user_id').tail(1).index
train_df = df_sorted.drop(last_idx)
test_df = df_sorted.loc[last_idx]

# Rebuild train matrix
rows = train_df['user_id'].map(user_map)
cols = train_df['game'].map(item_map)
vals = np.log1p(train_df['bet_amount']).to_numpy()
mat_train = sp.coo_matrix((vals, (rows, cols)), shape=mat.shape).tocsr()

# Retrain quickly on train set
U_t, V_t = train_als(mat_train, n_factors=12, n_iters=6, reg=0.1)

# Ground truth per user
truth = test_df.groupby('user_id')['game'].apply(list).to_dict()

def precision_at_k(U, V, mat_train, truth, k=5):
    precisions = []
    for u_id, u_index in user_map.items():
        known = set(mat_train[u_index].indices)
        rec = recommend_for_user(u_index, U, V, known_items=known, topk=k)
        true_games = truth.get(u_id, [])
        if not true_games: continue
        hit = sum([1 for g in rec if g in true_games])
        precisions.append(hit / k)
    return np.mean(precisions)

print('Precision@5:', precision_at_k(U_t, V_t, mat_train, truth, k=5))


iter 5/6
Precision@5: 0.20000000000000004


Predicor

In [13]:
# ======= Preparación: orden y masks (evita groupby.apply) =======
df_sorted = df.sort_values('timestamp').reset_index(drop=True)

# marcar el índice de la última interacción por usuario
last_per_user = df_sorted.groupby('user_id')['timestamp'].transform('max')
mask_prior = df_sorted['timestamp'] != last_per_user

# prior: todas las interacciones excepto la última por usuario
prior = df_sorted.loc[mask_prior].copy()

# test: las últimas interacciones (1 por usuario)
test_df = df_sorted.loc[~mask_prior].copy().reset_index(drop=True)

# ======= Agregados simples a partir de prior =======
agg = prior.groupby('user_id').agg({
    'bet_amount': ['mean', 'sum', 'max', 'std', 'count']
})
agg.columns = ['bet_mean', 'bet_sum', 'bet_max', 'bet_std', 'bet_count']
agg = agg.reset_index()

# rellenar NaN y valores problemáticos
agg['bet_std'] = agg['bet_std'].fillna(0)

# ======= Feature adicional: hora más frecuente (modo) y último juego =======
# hora modal (por si el usuario suele jugar a la misma hora)
prior['hour'] = prior['timestamp'].dt.hour
hour_mode = prior.groupby('user_id')['hour'].agg(lambda x: x.mode().iloc[0] if len(x.mode())>0 else 0).reset_index().rename(columns={'hour':'hour_mode'})

# último juego jugado antes de la última (puede usarse como strong feature)
last_before = prior.groupby('user_id').tail(1)[['user_id', 'game']].rename(columns={'game':'last_game'})

# merge features
features = agg.merge(hour_mode, on='user_id', how='left').merge(last_before, on='user_id', how='left').fillna(0)

# ======= Si tienes embeddings de ALS: añadirlos como features (opcional) =======
# suponiendo U_t (user factors) entrenado y user_map dict disponible
try:
    # U_t: numpy array (n_users x k)
    # user_map: dict user_id -> index
    user_emb_df = pd.DataFrame(U_t, index=list(user_map.keys()))
    user_emb_df = user_emb_df.reset_index().rename(columns={'index':'user_id'})
    user_emb_df.columns = ['user_id'] + [f'u_emb_{i}' for i in range(user_emb_df.shape[1]-1)]
    features = features.merge(user_emb_df, on='user_id', how='left').fillna(0)
except Exception:
    # si no existen embeddings, continuamos sin ellos
    pass

# ======= Preparar set final para entrenamiento: merge test_df con features =======
clf_df = test_df.merge(features, on='user_id', how='left').fillna(0)

# target y X
y = clf_df['game'].values
X = clf_df.drop(columns=['user_id', 'timestamp', 'game', 'date'], errors='ignore')  # elimina columnas no-features

# ======= Codificación de last_game (si está presente) =======
if 'last_game' in X.columns:
    X = pd.get_dummies(X, columns=['last_game'], prefix='last')

# Convertir a numpy
X_values = X.values

# ======= Split, escalado y modelo más robusto =======
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_values, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# RandomForest con class_weight para datos desbalanceados
clf_rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
clf_rf.fit(X_train_s, y_train)

y_pred = clf_rf.predict(X_test_s)
print("Accuracy:", clf_rf.score(X_test_s, y_test))
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))




Accuracy: 0.28
                    precision    recall  f1-score   support

          baccarat       0.10      0.06      0.07        35
         blackjack       0.32      0.55      0.41        42
      poker_3cards       0.27      0.19      0.22        42
poker_texas_holdem       0.24      0.23      0.23        40
           pontoon       0.33      0.34      0.34        41

          accuracy                           0.28       200
         macro avg       0.25      0.27      0.25       200
      weighted avg       0.26      0.28      0.26       200



Guardado

In [11]:
import joblib, os
os.makedirs('/content/artifacts', exist_ok=True)
joblib.dump({'U': U_t, 'V': V_t, 'user_map': user_map, 'item_map': item_map}, '/content/artifacts/als_factors.pkl')
joblib.dump({'clf': clf, 'scaler': scaler, 'le': le}, '/content/artifacts/classifier.pkl')
print('Artefactos guardados en /content/artifacts')
# Si montaste Drive, copia: !cp -r /content/artifacts /content/drive/MyDrive/


Artefactos guardados en /content/artifacts


Conclusiones


In [12]:
import numpy as np
import pandas as pd

np.random.seed(42)

games = ['blackjack','poker_texas_holdem','poker_cards','baccarat','pontoon']

n = 100

conclusiones_df = pd.DataFrame({
    'user_id': [f'user_{i}' for i in range(1, n+1)],
    'bet_mean': np.random.uniform(5, 80, n).round(2),
    'bet_max': np.random.uniform(20, 200, n).round(2),
    'interactions': np.random.randint(5, 60, n),
    'favorite_game': np.random.choice(games, n)
})

conclusiones_df.head(20)


,user_id,bet_mean,bet_max,interactions,favorite_game
0,user_1,33.09,25.66,28,poker_texas_holdem
1,user_2,76.30,134.55,56,poker_3cards
2,user_3,59.90,76.58,15,poker_texas_holdem
3,user_4,49.90,111.54,53,poker_texas_holdem
4,user_5,16.70,183.36,12,poker_texas_holdem
5,user_6,16.70,64.87,40,blackjack
6,user_7,9.36,93.87,42,blackjack
7,user_8,69.96,156.00,44,blackjack
8,user_9,50.08,61.18,24,poker_3cards
9,user_10,58.11,33.86,39,pontoon
